In [ ]:
# Lab type: extend
# Course: DS102 — Data Visualisation and Communication
# Lesson: From Exploration to Presentation
# Task: Audit a rough exploration chart, apply the chart junk checklist, add annotations, and build a presentation-ready figure using the presentation_chart helper.

# Lab: From Exploration to Presentation

This lab is the capstone for DS102. You'll apply the full workflow from Lesson 8:
audit a rough exploration chart, strip chart junk, add purposeful annotations,
and export a presentation-ready figure.

The lab is structured as a before/after exercise. Each step shows you an
under-polished chart first, then asks you to improve it.

**Outputs are cleared.** Run each cell to generate the plots.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install matplotlib seaborn pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import seaborn as sns

np.random.seed(42)
n = 500

df = pd.DataFrame({
    "date": pd.date_range("2023-01-01", periods=n, freq="D"),
    "category": np.random.choice(["Electronics", "Clothing", "Books", "Home"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "sales": np.random.lognormal(mean=6.5, sigma=0.8, size=n).round(2),
    "units": np.random.randint(1, 20, n),
    "customer_age": np.random.normal(38, 10, n).clip(18, 70).astype(int),
    "profit_margin": np.random.beta(3, 7, n).round(3),
})

monthly = df.set_index("date").resample("MS")["sales"].sum().reset_index()
monthly.columns = ["date", "total_sales"]

cat_sales = df.groupby("category")["sales"].sum().sort_values(ascending=False)

print(df.shape)

## Step 1: Audit an exploration chart

The cell below produces a rough exploration bar chart. Run it, then use the
chart junk audit from Lesson 8 to list every element that should be changed
before sharing this chart with a stakeholder.

In [ ]:
# --- BEFORE: rough exploration chart ---
fig, ax = plt.subplots()
ax.bar(cat_sales.index, cat_sales.values,
       color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])
ax.set_title("sales by category")
plt.show()

> **Audit checklist** — list every issue you can spot:
>
> - Title doesn't state a finding
> - *(add more issues here)*

*(Complete the list above before moving to Step 2.)*

## Step 2: Build the presentation version

Fix every issue you identified in Step 1. Requirements:

- Title states a specific finding
- Y-axis uses currency formatting (£ with comma thousands separator)
- Top and right spines removed
- Single colour (no rainbow bars)
- Axis labels with units
- Data source note added via `fig.text()`

In [ ]:
sns.set_theme(style="white", font_scale=1.0)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(cat_sales.index, cat_sales.values, color="#4c72b0")

ax.set_title("Electronics accounts for the highest total revenue", fontsize=13, pad=12)
ax.set_xlabel("Product category")
ax.set_ylabel("Total sales (£)")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.grid(axis="y", color="0.92", linewidth=0.8)

fig.text(0.12, -0.03, "Note: Retail transactions Jan–Dec 2023. N=500.",
         fontsize=8, color="0.5", ha="left")

fig.tight_layout()
plt.show()

> **Question:** Which changes made the biggest difference to readability?
> Is there anything you'd still change?

*(Write your answer here.)*

## Step 3: Annotate the peak month

Build a monthly sales line chart and use `ax.annotate` to mark the peak month
with an arrow and label. The annotation should state the value, not just the month.

In [ ]:
peak_row = monthly.loc[monthly["total_sales"].idxmax()]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly["date"], monthly["total_sales"], color="#4c72b0", linewidth=2)
ax.fill_between(monthly["date"], monthly["total_sales"], alpha=0.1, color="#4c72b0")

ax.annotate(
    f"Peak: £{peak_row['total_sales']:,.0f}",
    xy=(peak_row["date"], peak_row["total_sales"]),
    xytext=(peak_row["date"], peak_row["total_sales"] * 1.12),
    ha="center",
    fontsize=9,
    color="#4c72b0",
    arrowprops=dict(arrowstyle="->", color="#4c72b0", lw=1.2),
)

ax.set_title("Monthly sales peaked in Q4", fontsize=13)
ax.set_ylabel("Total sales (£)")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.grid(axis="y", color="0.92", linewidth=0.8)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
plt.show()

> **Question:** Does the annotation make the finding more or less immediately clear
> than a title alone? Try changing `xytext` to move the label — where does it work
> best visually?

*(Write your answer here.)*

## Step 4: Export the chart

Save the annotated monthly sales chart to disk as both a PNG (for reports)
and an SVG (for vector use). Check the output files were created.

In [ ]:
import os

fig.savefig("monthly_sales.png", dpi=150, bbox_inches="tight", facecolor="white")
fig.savefig("monthly_sales.svg", bbox_inches="tight")

for fname in ["monthly_sales.png", "monthly_sales.svg"]:
    size_kb = os.path.getsize(fname) / 1024
    print(f"{fname}: {size_kb:.1f} KB")

> **Question:** How large is the PNG versus the SVG?
> In what situation would you choose SVG over PNG?

*(Write your answer here.)*

## Challenge: build a presentation chart from scratch

Using the `presentation_chart` helper from Lesson 8, build a chart that answers
the following question for a stakeholder:

> *"Which regions are driving the most units sold, and has that changed over the year?"*

Requirements:
- Use the `presentation_chart` helper below
- Choose an appropriate chart type (line? bar? small multiples?)
- Title must state the key finding
- Y-axis must be labelled with units
- Export as PNG
- The final checklist from Lesson 8 must all pass

In [ ]:
def presentation_chart(title, note=None):
    """Returns fig, ax with presentation-ready defaults applied."""
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.grid(axis="y", color="0.92", linewidth=0.8)
    ax.set_title(title, fontsize=13, pad=12)
    if note:
        fig.text(0.12, -0.03, f"Note: {note}", fontsize=8, color="0.5", ha="left")
    return fig, ax

In [ ]:
# Aggregate: monthly units by region
monthly_units = (
    df.set_index("date")
    .groupby("region")
    .resample("MS")["units"]
    .sum()
    .reset_index()
)

# Build your chart here using presentation_chart()


## Final checklist

Before you sign off on a chart, go through this list.
Check each item for the chart you built in the challenge above.

- [ ] Title states the finding, not the variable
- [ ] Both axes labelled with units
- [ ] Numbers formatted (currency, %, commas for thousands)
- [ ] Top and right spines removed
- [ ] Colour encodes something meaningful, not decoration
- [ ] Chart passes the greyscale test (desaturate mentally or in an image editor)
- [ ] Data source noted
- [ ] Exported with `dpi=150`, `bbox_inches="tight"`, `facecolor="white"`